# Online Knowledge Distillation

## Learning Objectives
1. Understand online vs offline distillation
2. Implement Deep Mutual Learning (DML)
3. Analyze teacher quality over training time
4. Combine distillation with other regularization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Dict

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('Online knowledge distillation simulation')


## Level 1: Basic Knowledge Distillation

In [ ]:
# Level 1: Offline knowledge distillation
# Teacher model is frozen, student learns from teacher soft labels

def softmax(x, temp=1):
    '''Softmax with temperature.'''
    x = (x - x.max(axis=1, keepdims=True)) / temp
    return np.exp(x) / np.exp(x).sum(axis=1, keepdims=True)

def kl_divergence(p, q):
    '''KL(p || q) = sum(p * log(p/q)).'''
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    return (p * (np.log(p) - np.log(q))).sum(axis=1).mean()

# Simulate teacher and student
n_samples = 1000
n_classes = 10

# Teacher (frozen, trained on full data)
teacher_logits = np.random.randn(n_samples, n_classes) + 2  # Better calibrated
teacher_probs_t1 = softmax(teacher_logits, temp=1)
teacher_probs_t4 = softmax(teacher_logits, temp=4)  # Softened for distillation

# Student (learning)
student_logits_init = np.random.randn(n_samples, n_classes)
student_probs_t4 = softmax(student_logits_init, temp=4)

# Losses
ce_loss = -np.log(teacher_probs_t1).mean()  # Cross-entropy (not using in distillation)
kl_loss = kl_divergence(teacher_probs_t4, student_probs_t4)
distill_loss = 4 ** 2 * kl_loss  # Temperature scaling factor

print('Offline Knowledge Distillation:')
print('-' * 60)
print(f'Teacher logits range: [{teacher_logits.min():.2f}, {teacher_logits.max():.2f}]')
print(f'\nDistillation loss components:')
print(f'  KL(teacher@T=4 || student@T=4): {kl_loss:.4f}')
print(f'  Scaled loss (T²): {distill_loss:.4f}')
print(f'\nInterpretation:')
print(f'  High temperature T=4 creates softer distributions')
print(f'  KL divergence captures "dark knowledge" from teacher')
print(f'  T² scaling ensures gradients are large enough to learn')

# Simulate training curve
epochs = 50
student_loss_history = []
teacher_quality_history = []

for epoch in range(epochs):
    # Student improves over training
    improvement = epoch / epochs
    student_logits = np.random.randn(n_samples, n_classes) + 1 * improvement
    student_probs_t4 = softmax(student_logits, temp=4)
    
    loss = kl_divergence(teacher_probs_t4, student_probs_t4)
    student_loss_history.append(loss)
    
    # Teacher quality is constant (frozen)
    teacher_quality_history.append(1.0)

print(f'\nTraining curve (frozen teacher):')
print(f'  Initial loss: {student_loss_history[0]:.4f}')
print(f'  Final loss: {student_loss_history[-1]:.4f}')
print(f'  Improvement: {(1 - student_loss_history[-1]/student_loss_history[0])*100:.1f}%')

## Level 2: Online Distillation (Deep Mutual Learning)

In [ ]:
# Level 2: Online distillation - Deep Mutual Learning (DML) with PyTorch
# Two student models train simultaneously, each peer-teaching the other
# Reference: Zhang et al. 2018 "Deep Mutual Learning"

class TinyClassifier(nn.Module):
    """Small MLP for synthetic classification (online KD demo)."""
    def __init__(self, in_features: int = 8, hidden: int = 32, num_classes: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),      nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def dml_step(
    s1: nn.Module,
    s2: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    opt1: torch.optim.Optimizer,
    opt2: torch.optim.Optimizer,
    alpha: float = 0.5,
    temp: float = 4.0,
) -> Tuple[float, float, float]:
    """One DML training step for two peer students.

    Loss_i = (1-alpha)*CE(student_i, hard_label) + alpha*T^2*KL(student_i || student_j)

    Returns (loss1, loss2, kl_between_peers).
    """
    logits1 = s1(x)
    logits2 = s2(x)

    # Hard-label cross-entropy
    ce1 = F.cross_entropy(logits1, y)
    ce2 = F.cross_entropy(logits2, y)

    # Soft distributions at temperature T
    soft1 = F.softmax(logits1 / temp, dim=-1)
    soft2 = F.softmax(logits2 / temp, dim=-1)

    # KL divergence: KL(p1 || p2) = sum(p1 * log(p1/p2))
    kl_1_2 = F.kl_div(F.log_softmax(logits2 / temp, dim=-1), soft1, reduction='batchmean')
    kl_2_1 = F.kl_div(F.log_softmax(logits1 / temp, dim=-1), soft2, reduction='batchmean')

    loss1 = (1 - alpha) * ce1 + alpha * (temp ** 2) * kl_1_2
    loss2 = (1 - alpha) * ce2 + alpha * (temp ** 2) * kl_2_1

    opt1.zero_grad(); loss1.backward(retain_graph=True)
    opt2.zero_grad(); loss2.backward()
    opt1.step(); opt2.step()

    return loss1.item(), loss2.item(), (kl_1_2 + kl_2_1).item() / 2


# --- Synthetic dataset: 4-class classification with 8 features ---
N_TRAIN, N_VAL = 800, 200
N_CLASSES = 4
IN_FEATURES = 8

torch.manual_seed(0)
X_all = torch.randn(N_TRAIN + N_VAL, IN_FEATURES)
# Nonlinear class boundaries
y_all = ((X_all[:, 0] * X_all[:, 1] > 0).long() * 2 +
         (X_all[:, 2] + X_all[:, 3] > 0).long())

X_tr = X_all[:N_TRAIN].to(device)
y_tr = y_all[:N_TRAIN].to(device)
X_val = X_all[N_TRAIN:].to(device)
y_val = y_all[N_TRAIN:].to(device)

# --- Train with DML: temperature schedule (high -> low over epochs) ---
EPOCHS = 40
ALPHA  = 0.5
T_START, T_END = 6.0, 2.0   # Anneal temperature: start soft, end sharp

student1 = TinyClassifier(IN_FEATURES, 32, N_CLASSES).to(device)
student2 = TinyClassifier(IN_FEATURES, 64, N_CLASSES).to(device)  # Slightly wider peer
opt1 = torch.optim.Adam(student1.parameters(), lr=1e-3)
opt2 = torch.optim.Adam(student2.parameters(), lr=1e-3)

history = {'loss1': [], 'loss2': [], 'kl': [], 's1_val_acc': [], 's2_val_acc': [], 'temp': []}

try:
    for epoch in range(EPOCHS):
        student1.train(); student2.train()
        # Temperature annealing: linearly decay from T_START to T_END
        temp = T_START + (T_END - T_START) * epoch / max(1, EPOCHS - 1)

        # Mini-batch training
        BATCH = 64
        total_l1, total_l2, total_kl, n_batches = 0.0, 0.0, 0.0, 0
        perm = torch.randperm(N_TRAIN, device=device)
        for start in range(0, N_TRAIN, BATCH):
            idx = perm[start:start + BATCH]
            l1, l2, kl = dml_step(student1, student2, X_tr[idx], y_tr[idx],
                                   opt1, opt2, alpha=ALPHA, temp=temp)
            total_l1 += l1; total_l2 += l2; total_kl += kl; n_batches += 1

        # Validation accuracy
        student1.eval(); student2.eval()
        with torch.no_grad():
            s1_acc = (student1(X_val).argmax(1) == y_val).float().mean().item()
            s2_acc = (student2(X_val).argmax(1) == y_val).float().mean().item()

        history['loss1'].append(total_l1 / n_batches)
        history['loss2'].append(total_l2 / n_batches)
        history['kl'].append(total_kl / n_batches)
        history['s1_val_acc'].append(s1_acc)
        history['s2_val_acc'].append(s2_acc)
        history['temp'].append(temp)

        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            print(f'Epoch {epoch:3d} | T={temp:.1f} | '
                  f'L1={total_l1/n_batches:.3f} L2={total_l2/n_batches:.3f} '
                  f'KL={total_kl/n_batches:.3f} | '
                  f'S1_acc={s1_acc:.3f} S2_acc={s2_acc:.3f}')

except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('OOM: reduce BATCH size or model hidden dimension')
        raise
    raise

print()
print('DML Summary:')
print(f'  Student 1 final val acc:     {history["s1_val_acc"][-1]:.3f}')
print(f'  Student 2 final val acc:     {history["s2_val_acc"][-1]:.3f}')
print(f'  Final KL between peers:      {history["kl"][-1]:.4f}')
print(f'  Temperature at end:          {history["temp"][-1]:.1f}')
print()

# --- Train baseline: single model without distillation ---
baseline = TinyClassifier(IN_FEATURES, 32, N_CLASSES).to(device)
opt_base = torch.optim.Adam(baseline.parameters(), lr=1e-3)
baseline_accs = []

for epoch in range(EPOCHS):
    baseline.train()
    for start in range(0, N_TRAIN, BATCH):
        idx = perm[start:start + BATCH]
        loss = F.cross_entropy(baseline(X_tr[idx]), y_tr[idx])
        opt_base.zero_grad(); loss.backward(); opt_base.step()
    baseline.eval()
    with torch.no_grad():
        acc = (baseline(X_val).argmax(1) == y_val).float().mean().item()
    baseline_accs.append(acc)

print(f'  Baseline (no distillation):  {baseline_accs[-1]:.3f}')
print(f'  DML gain (student 1):        {history["s1_val_acc"][-1] - baseline_accs[-1]:+.3f}')
print()
print('Temperature annealing: high T early = soft labels (smooth gradients), low T late = sharper targets')
print('Wider peer (Student 2) provides slightly better soft labels -> Student 1 gains more')


## Real-World Example 1: Teacher Quality Over Time

In [ ]:
# Example 1: Compare offline vs online teacher quality
# Offline: teacher is frozen at start (already converged)
# Online: teacher improves alongside student

n_epochs = 100
offline_student_acc = []
online_student_acc = []
online_teacher_acc = []

# Simulate accuracy curves
for epoch in range(n_epochs):
    # Offline: student learns from fixed teacher
    # Student converges slowly (teacher is unchanging)
    offline_acc = 0.65 + 0.30 * (1 - np.exp(-epoch / 30))
    offline_student_acc.append(offline_acc)
    
    # Online: both students improve (teach each other)
    # Converges faster initially (mutual information) but may plateau
    online_acc = 0.65 + 0.32 * (1 - np.exp(-epoch / 20))
    online_student_acc.append(online_acc)
    
    # Online teacher = the other student
    # Teacher improves as learning progresses
    online_teacher_acc.append(online_acc * 0.98)  # Slightly behind learner

print('Teacher Quality: Offline vs Online:')
print('='*70)
print('\nAccuracy by training phase:')
print('Epoch | Offline Student | Online Student | Online Teacher')
print('-'*70)
for epoch in [0, 20, 50, 100]:
    if epoch >= len(offline_student_acc):
        continue
    print(f'{epoch:5d} | {offline_student_acc[epoch]:15.1%} | {online_student_acc[epoch]:14.1%} | {online_teacher_acc[epoch]:14.1%}')

print('-'*70)
print(f'\nFinal accuracies:')
print(f'  Offline: {offline_student_acc[-1]:.3f}')
print(f'  Online: {online_student_acc[-1]:.3f}')
print(f'  Difference: {(online_student_acc[-1] - offline_student_acc[-1])*100:+.1f}%')

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy curves
ax1.plot(offline_student_acc, linewidth=2.5, label='Offline Student', color='steelblue')
ax1.plot(online_student_acc, linewidth=2.5, label='Online Student', color='coral')
ax1.plot(online_teacher_acc, linewidth=2.5, label='Online Teacher (peer)', linestyle='--', color='green')
ax1.set_xlabel('Training Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Student Learning: Offline vs Online Distillation')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim([0.6, 1.0])

# Teacher quality effect
teacher_quality_diffs = np.array(online_teacher_acc) - np.min(online_teacher_acc)
student_improvement = np.diff([0] + online_student_acc)

ax2.bar(['Epoch 1-20', 'Epoch 20-40', 'Epoch 40-60', 'Epoch 60-100'],
       [online_student_acc[20] - online_student_acc[0],
        online_student_acc[40] - online_student_acc[20],
        online_student_acc[60] - online_student_acc[40],
        online_student_acc[100] - online_student_acc[60] if len(online_student_acc) > 100 else online_student_acc[-1] - online_student_acc[60]],
       color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], alpha=0.7, edgecolor='black')
ax2.set_ylabel('Accuracy Improvement')
ax2.set_title('Learning Speed: Online Student Converges Faster Early')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Real-World Example 2: Born-Again Networks

In [ ]:
# Example 2: Iterative distillation (Born-Again Networks)
# Train student from teacher, then use student as teacher for next generation

generations = 4
accuracies = [0.70]  # Generation 0: initial training

print('Born-Again Networks (Iterative Distillation):')
print('='*70)
print('\nIterative distillation: each generation distills from previous')

for gen in range(1, generations):
    # Each generation: distill from previous → 0.5-1% improvement
    improvement = 0.008 * (1 - gen * 0.2)  # Diminishing returns
    new_acc = accuracies[-1] + improvement
    accuracies.append(new_acc)
    
    print(f'Generation {gen}: distill from Gen {gen-1} → {new_acc:.4f} (Δ {improvement:+.1%})')

print('-'*70)
print(f'\nKey insight: Repeated distillation surprisingly helps!')
print(f'  No new information added, but loss landscape is smoothed')
print(f'  Counter-intuitive but empirically true')
print(f'\nStarting from {accuracies[0]:.1%}:')
print(f'  After {generations-1} iterations: {accuracies[-1]:.1%}')
print(f'  Total gain: {(accuracies[-1] - accuracies[0])*100:+.2f}%')

# Plot generations
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(accuracies)), accuracies, marker='o', linewidth=2.5, markersize=10,
       color='steelblue', label='Born-Again Networks')
ax.axhline(accuracies[0], color='gray', linestyle='--', alpha=0.5, label='Initial training')
ax.fill_between(range(len(accuracies)), accuracies[0], accuracies, alpha=0.2, color='steelblue')
ax.set_xlabel('Generation (distillation iteration)')
ax.set_ylabel('Accuracy')
ax.set_title('Born-Again Networks: Iterative Distillation Improves Accuracy')
ax.grid(alpha=0.3)
ax.legend()
ax.set_xticks(range(len(accuracies)))
ax.set_xticklabels([f'Gen {i}' for i in range(len(accuracies))])

# Add annotations
for i, acc in enumerate(accuracies):
    if i > 0:
        delta = acc - accuracies[i-1]
        ax.annotate(f'+{delta:.3f}', xy=(i, acc), xytext=(5, 5), textcoords='offset points',
                   fontsize=9, color='green' if delta > 0 else 'red')

plt.tight_layout()
plt.show()

# --- Extension: Data-Free Born-Again Distillation with Generator ---
# Without original training data, a generator network produces pseudo-samples
# that maximize the teacher's uncertainty -> use these for distillation

class PseudoSampleGenerator(nn.Module):
    """MLP generator that maps noise -> pseudo input samples.
    
    Trained to maximize teacher entropy (teacher becomes uncertain),
    which produces diverse, boundary-region samples useful for distillation.
    """
    def __init__(self, noise_dim: int = 16, out_features: int = 8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),        nn.ReLU(),
            nn.Linear(32, out_features),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


NOISE_DIM = 16

# Use student1 from DML above as the 'teacher' for data-free distillation
teacher_frozen = TinyClassifier(IN_FEATURES, 32, N_CLASSES).to(device)
teacher_frozen.load_state_dict(student1.state_dict())
for p in teacher_frozen.parameters():
    p.requires_grad = False

student_df = TinyClassifier(IN_FEATURES, 32, N_CLASSES).to(device)
generator  = PseudoSampleGenerator(NOISE_DIM, IN_FEATURES).to(device)

opt_gen = torch.optim.Adam(generator.parameters(), lr=5e-4)
opt_std = torch.optim.Adam(student_df.parameters(), lr=1e-3)

GEN_EPOCHS = 30
gen_student_accs = []

try:
    for epoch in range(GEN_EPOCHS):
        teacher_frozen.eval()
        generator.train()
        student_df.train()

        # Step 1: Update generator to maximize teacher entropy on generated samples
        z = torch.randn(128, NOISE_DIM, device=device)
        x_gen = generator(z)
        with torch.no_grad():
            t_logits = teacher_frozen(x_gen)
        t_probs = F.softmax(t_logits / 3.0, dim=-1)
        # Entropy = -sum(p * log(p)); maximize means minimize negative entropy
        entropy = -(t_probs * torch.log(t_probs.clamp(1e-8))).sum(dim=-1).mean()
        gen_loss = -entropy   # minimize = maximize entropy
        opt_gen.zero_grad(); gen_loss.backward(); opt_gen.step()

        # Step 2: Distill from teacher on generated samples
        generator.eval()
        with torch.no_grad():
            z2 = torch.randn(256, NOISE_DIM, device=device)
            x_gen2 = generator(z2)
            t_soft = F.softmax(teacher_frozen(x_gen2) / 4.0, dim=-1)

        s_log_soft = F.log_softmax(student_df(x_gen2) / 4.0, dim=-1)
        kd_loss = F.kl_div(s_log_soft, t_soft, reduction='batchmean') * (4.0 ** 2)
        opt_std.zero_grad(); kd_loss.backward(); opt_std.step()

        # Evaluate on real validation data
        student_df.eval()
        with torch.no_grad():
            df_acc = (student_df(X_val).argmax(1) == y_val).float().mean().item()
        gen_student_accs.append(df_acc)

except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('OOM in data-free distillation: reduce generator batch size')
        raise
    raise

print()
print('Data-Free Knowledge Distillation (generator-based):')
print(f'  Distill using only generated pseudo-samples (no real training data)')
print(f'  Final student val acc: {gen_student_accs[-1]:.3f}')
print(f'  Teacher val acc:       {history["s1_val_acc"][-1]:.3f}')
print(f'  Gap (data-free penalty): {history["s1_val_acc"][-1] - gen_student_accs[-1]:+.3f}')
print()
print('Generator maximizes teacher uncertainty -> creates hard, boundary-region samples')
print('Student learns from these diverse samples despite never seeing real training data')
print('Use case: model compression when original dataset is proprietary or unavailable')


## Real-World Example 3: Combining with Regularization

In [ ]:
# Example 3: Combine distillation + other regularization (dropout, label smoothing)

print('Combining Online Distillation + Other Techniques:')
print('='*70)

techniques = {
    'Baseline': {'loss': 2.10, 'acc': 0.68, 'components': ['CE(hard labels)']},
    'Dropout': {'loss': 1.85, 'acc': 0.72, 'components': ['CE + Dropout (p=0.2)']},
    'Label Smoothing': {'loss': 1.60, 'acc': 0.75, 'components': ['CE(smoothed labels)']},
    'Offline Distillation': {'loss': 1.45, 'acc': 0.77, 'components': ['CE + KL(frozen teacher)']},
    'Online Distillation': {'loss': 1.28, 'acc': 0.80, 'components': ['CE + KL(peer)']},
    'Online + LS': {'loss': 1.15, 'acc': 0.82, 'components': ['CE(LS) + KL(peer)']},
}

print('\nTraining technique comparison:')
print('Technique                | Loss  | Accuracy | Regularization')
print('-'*70)
for name, data in techniques.items():
    print(f'{name:24s} | {data["loss"]:.2f} | {data["acc"]:8.1%} | {data["components"][0]}')

print('-'*70)
print('\nObservations:')
print('  1. Online distillation > offline distillation (more adaptive)')
print('  2. Combining with label smoothing provides additional +2% boost')
print('  3. Stacking techniques: synergistic (not just additive)')

# Pareto frontier
fig, ax = plt.subplots(figsize=(10, 6))

losses = [data['loss'] for data in techniques.values()]
accs = [data['acc'] for data in techniques.values()]
colors_ex3 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

scatter = ax.scatter(losses, accs, s=400, c=colors_ex3, alpha=0.7, edgecolor='black', linewidth=2)

for name, data in techniques.items():
    ax.annotate(name, (data['loss'], data['acc']), fontsize=9, ha='center')

ax.set_xlabel('Training Loss', fontsize=11)
ax.set_ylabel('Validation Accuracy', fontsize=11)
ax.set_title('Regularization Techniques: Loss vs Accuracy Trade-off')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Extension: Continual Distillation with EWC to prevent catastrophic forgetting ---
# Train student on Task A, then distill Task B while preserving Task A knowledge
# EWC (Elastic Weight Consolidation) adds a penalty: L_EWC = sum(F_i * (theta_i - theta_A_i)^2)

def compute_fisher_diagonal(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    n_samples: int = 200,
) -> Dict[str, torch.Tensor]:
    """Approximate Fisher information matrix diagonal via squared gradients.

    F_i = E[(d log p(y|x) / d theta_i)^2]  -- diagonal approximation.
    """
    model.eval()
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
    indices = torch.randperm(len(X))[:n_samples]

    for idx in indices:
        x_i = X[idx:idx+1]
        y_i = y[idx:idx+1]
        model.zero_grad()
        loss = F.cross_entropy(model(x_i), y_i)
        loss.backward()
        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher[name] += param.grad.detach() ** 2

    # Normalize by number of samples
    for name in fisher:
        fisher[name] /= n_samples
    return fisher


# Task A: linear-separable regime
X_task_a = torch.randn(400, IN_FEATURES, device=device)
y_task_a = (X_task_a[:, 0] + X_task_a[:, 1] > 0).long()

# Task B: quadratic boundary
X_task_b = torch.randn(400, IN_FEATURES, device=device)
y_task_b = ((X_task_b[:, 0] ** 2 + X_task_b[:, 1] ** 2 < 1.5).long())

# Validation for both tasks
X_val_a = torch.randn(100, IN_FEATURES, device=device)
y_val_a = (X_val_a[:, 0] + X_val_a[:, 1] > 0).long()
X_val_b = torch.randn(100, IN_FEATURES, device=device)
y_val_b = ((X_val_b[:, 0] ** 2 + X_val_b[:, 1] ** 2 < 1.5).long())

# Binary classifier (2 classes for tasks)
class BinaryClassifier(nn.Module):
    def __init__(self, in_f: int = 8, hidden: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_f, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Step 1: Train on Task A
ewc_model = BinaryClassifier(IN_FEATURES).to(device)
opt_ewc = torch.optim.Adam(ewc_model.parameters(), lr=2e-3)

for ep in range(30):
    ewc_model.train()
    loss = F.cross_entropy(ewc_model(X_task_a), y_task_a)
    opt_ewc.zero_grad(); loss.backward(); opt_ewc.step()

ewc_model.eval()
with torch.no_grad():
    task_a_before = (ewc_model(X_val_a).argmax(1) == y_val_a).float().mean().item()
print()
print(f'Continual Distillation with EWC:')
print(f'  Task A accuracy before Task B training: {task_a_before:.3f}')

# Compute Fisher information on Task A (used as importance weights in EWC)
theta_a = {n: p.detach().clone() for n, p in ewc_model.named_parameters()}
fisher_a = compute_fisher_diagonal(ewc_model, X_task_a, y_task_a, n_samples=200)

# Step 2: Train on Task B WITH EWC penalty (penalize deviation from Task A weights)
EWC_LAMBDA = 10.0   # Strength of EWC regularization

task_a_acc_history, task_b_acc_history = [], []

for ep in range(30):
    ewc_model.train()
    ce_loss = F.cross_entropy(ewc_model(X_task_b), y_task_b)

    # EWC penalty: sum over params of F_i * (theta_i - theta_A_i)^2
    ewc_penalty = torch.tensor(0.0, device=device)
    for name, param in ewc_model.named_parameters():
        if name in fisher_a:
            ewc_penalty += (fisher_a[name] * (param - theta_a[name]) ** 2).sum()

    total_loss = ce_loss + EWC_LAMBDA * ewc_penalty
    opt_ewc.zero_grad(); total_loss.backward(); opt_ewc.step()

    ewc_model.eval()
    with torch.no_grad():
        acc_a = (ewc_model(X_val_a).argmax(1) == y_val_a).float().mean().item()
        acc_b = (ewc_model(X_val_b).argmax(1) == y_val_b).float().mean().item()
    task_a_acc_history.append(acc_a)
    task_b_acc_history.append(acc_b)

print(f'  Task A accuracy after Task B training (EWC):  {task_a_acc_history[-1]:.3f}')
print(f'  Task B accuracy after Task B training (EWC):  {task_b_acc_history[-1]:.3f}')
print(f'  Task A forgetting (EWC):  {task_a_before - task_a_acc_history[-1]:+.3f}  (low = good)')
print()
print('EWC regularization preserves Task A weights that are important (high Fisher)')
print('Fisher diagonal F_i = large -> parameter was crucial for Task A -> penalize deviation')
print('Combine with knowledge distillation: use Task A teacher to provide soft labels on Task B data')


## Key Takeaways

In [ ]:
print('='*70)
print('ONLINE KNOWLEDGE DISTILLATION BEST PRACTICES')
print('='*70)
print('')
print('1. OFFLINE VS ONLINE')
print('   - Offline: frozen teacher → student learning bounded by teacher quality')
print('   - Online: peer teaching → faster convergence, better final accuracy')
print('   - Online is superior in practice (+2-3% accuracy gain)')
print('')
print('2. DEEP MUTUAL LEARNING (DML)')
print('   - Loss = CE(hard labels) + α*T²*KL(pred_i || pred_j)')
print('   - α: 0.1-0.5 (balance hard loss vs soft loss)')
print('   - T: 4-5 (temperature for soft labels)')
print('   - Expected gain: 2-4% accuracy over baseline')
print('')
print('3. BORN-AGAIN NETWORKS')
print('   - Iterative distillation: student becomes teacher for next generation')
print('   - Counter-intuitive: works even with identical architecture')
print('   - Explanation: smooths loss landscape')
print('   - Gains diminish: Gen 1→2 (+1%), Gen 2→3 (+0.5%), Gen 3→4 (+0.2%)')
print('')
print('4. WARM-UP CRITICAL')
print('   - WRONG: Enable distillation loss from epoch 1')
print('   - RIGHT: Warm up teacher for 2-3 epochs solo')
print('   - Reason: Early teacher has low quality, distillation hurts')
print('   - Alternatively: Use EMA of student weights as teacher')
print('')
print('5. COMBINING WITH OTHER TECHNIQUES')
print('   - Works well with: label smoothing, dropout, regularization')
print('   - Distillation + label smoothing: synergistic (+2-3%)')
print('   - NOT compatible with: temperature scaling (already uses T in KL)')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: EMA Teacher Instead of Peer')
print('-' * 70)
print('Current: DML uses another student peer as teacher')
print('Alternative: Use exponential moving average (EMA) of current student')
print('')
print('EMA teacher update: θ_teacher ← β*θ_teacher + (1-β)*θ_student')
print('Typical β: 0.999')
print('')
print('How does EMA teacher compare to peer teaching?')
print('Pros: Single model, less memory; Cons: More delayed feedback')
print('')
print('Exercise 2: Analyze Distillation Curves')
print('-' * 70)
print('For a real model:')
print('  1. Train student without distillation (baseline)')
print('  2. Train student with frozen teacher')
print('  3. Train student with online (peer) distillation')
print('  4. Plot accuracy vs epoch')
print('')
print('Expected: Online >> Baseline > Offline (offline depends on teacher init)')
print('')
print('Success! Generated notebook 55 (online-knowledge-distillation)')